Set env and make link for easier access

In [1]:
%env PIP_CACHE_DIR="/home/pipcache"
%env TOTALSEG_HOME_DIR=/home/pipcache/.totalsegmentator
!ln -sf /Public ~

env: PIP_CACHE_DIR="/home/pipcache"
env: TOTALSEG_HOME_DIR=/home/pipcache/.totalsegmentator


Check env

In [2]:
!echo $PIP_CACHE_DIR
!echo $TOTALSEG_HOME_DIR

"/home/pipcache"
/home/pipcache/.totalsegmentator


In [ ]:
# 資料集影像在這裡展示的是牙齒的影像以及牙齒切割後的label影像
from pathlib import Path
import SimpleITK as sitk
import matplotlib.pyplot as plt

# 以使用者家目錄為基準
base_dir = Path.home()

# 使用相對於家目錄的路徑
image_path1 = base_dir / "Public/train/ToothFairy3/imagesTr/ToothFairy3F_001_0000.nii.gz"
image_path2 = base_dir / "Public/train/ToothFairy3/labelsTr/ToothFairy3F_001.nii.gz"

# 讀取影像
image1 = sitk.ReadImage(str(image_path1))
volume1 = sitk.GetArrayFromImage(image1)

image2 = sitk.ReadImage(str(image_path2))
volume2 = sitk.GetArrayFromImage(image2)

# 使用相同的 Z 切片
z = min(volume1.shape[0], volume2.shape[0]) // 2

# 並排顯示
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(volume1[z], cmap="gray", origin="lower")
axes[0].set_title("Original CT")
axes[0].axis("off")

axes[1].imshow(volume2[z], cmap="gray", origin="lower")
axes[1].set_title("Segmentation Label")
axes[1].axis("off")

plt.tight_layout()
plt.show()

Install Totalsegmentator

Segment body parts
set --device gpu: (0-7)

In [ ]:
import sys
import subprocess
import os
import hashlib

# 依帳號選 GPU（0-7 均勻分佈）；設定 COURSE_GPU 可固定
#GPU = int(
    #os.environ.get("COURSE_GPU")
    #or int(hashlib.sha256((os.environ.get("USER") or "shared").encode()).hexdigest(), 16) % 8
#)

#改成你想要使用的 GPU 編號，例如 0、1、2、3 等
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "3" #改成你想要使用的 GPU 編號，例如 0、1、2、3 等

cmd = [
    f"{sys.prefix}/bin/TotalSegmentator",
    "--device", "gpu",
    "-i", "/Public/inference/colon/1.3.6.1.4.1.9328.50.4.0001.nii.gz",  #胃部的影像來切割
    "-o", "segmentations",
]

print("Using:", cmd[0])
print("Physical GPU:", env["CUDA_VISIBLE_DEVICES"])
subprocess.run(cmd, check=True)

Using: /home/sandy0317/practice_seg/.pixi/envs/default/bin/TotalSegmentator


100%|██████████| 27/27 [00:02<00:00, 11.60it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 8.65s
Predicting part 1 of 5 ...


100%|██████████| 27/27 [00:01<00:00, 22.45it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 8.65s
Predicting part 1 of 5 ...
Predicting part 2 of 5 ...


...

100%|██████████| 27/27 [00:01<00:00, 22.63it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 8.65s
Predicting part 1 of 5 ...
Predicting part 2 of 5 ...
Predicting part 3 of 5 ...
Predicting part 4 of 5 ...


100%|██████████| 27/27 [00:01<00:00, 23.10it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 8.65s
Predicting part 1 of 5 ...
Predicting part 2 of 5 ...
Predicting part 3 of 5 ...
Predicting part 4 of 5 ...
Predicting part 5 of 5 ...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Resampling...
  Resampled in 8.65s
Predicting part 1 of 5 ...
Predicting part 2 of 5 ...
Predicting part 3 of 5 ...
Predicting part 4 of 5 ...
Predicting part 5 of 5 ...
  Predicted in 96.49s
Resampling...
  Resampled in 3.68s
Saving segmentations...
  Saved in 190.60s


CompletedProcess(args=['/home/sandy0317/practice_seg/.pixi/envs/default/bin/TotalSegmentator', '--device', 'gpu:3', '-i', '/Public/inference/colon/1.3.6.1.4.1.9328.50.4.0001.nii.gz', '-o', 'segmentations'], returncode=0)

In [ ]:
#查看切割結果
from pathlib import Path
import SimpleITK as sitk
import matplotlib.pyplot as plt

# 以使用者家目錄為基準
base_dir = Path.home()

# 兩張影像的路徑
image_path1 = base_dir / "Public/inference/colon/1.3.6.1.4.1.9328.50.4.0001.nii.gz"
image_path2 = Path("segmentations/colon.nii.gz")

# 檢查路徑
print("目前工作目錄：", Path.cwd())
print("Image 1：", image_path1, image_path1.exists())
print("Image 2：", image_path2.resolve(), image_path2.exists())

# 先確認檔案存在，再讀取
if not image_path1.exists():
    raise FileNotFoundError(f"找不到第一張影像：{image_path1}")

if not image_path2.exists():
    raise FileNotFoundError(f"找不到第二張影像：{image_path2.resolve()}")

volume1 = sitk.GetArrayFromImage(sitk.ReadImage(str(image_path1)))
volume2 = sitk.GetArrayFromImage(sitk.ReadImage(str(image_path2)))

# 顯示兩張灰階影像
z1 = volume1.shape[0] // 2
z2 = volume2.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(volume1[z1], cmap="gray", origin="lower")
axes[0].set_title(f"ToothFairy3 (Z={z1})")
axes[0].axis("off")

axes[1].imshow(volume2[z2], cmap="gray", origin="lower")
axes[1].set_title(f"FDI 37 (Z={z2})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

Segment teeth

In [ ]:
import sys
import subprocess
import os
import hashlib

# 依帳號選 GPU；teeth 用下一顆卡，把兩個 inference 分散
#GPU = int(
    #os.environ.get("COURSE_GPU")
    #or int(hashlib.sha256((os.environ.get("USER") or "shared").encode()).hexdigest(), 16) % 8
#)
#TEETH_GPU = (GPU + 1) % 8

#改成你想要使用的 GPU 編號，例如 0、1、2、3 等
env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "3"

cmd = [
    f"{sys.prefix}/bin/TotalSegmentator",
    "--device", "gpu",#f"gpu:{TEETH_GPU}"
    "-i", "/Public/inference/ToothFairy3/ToothFairy3F_001_0000.nii.gz",  #在這裡使用的是牙齒影像來切割
    "-o", "teeth",
    "--task", "teeth"
]

print("Using:", cmd[0])
#print("GPU:", TEETH_GPU)
print("Physical GPU:", env["CUDA_VISIBLE_DEVICES"])
subprocess.run(cmd, check=True)

Using: /home/sandy0317/practice_seg/.pixi/envs/default/bin/TotalSegmentator


100%|██████████| 1/1 [00:00<00:00,  1.83it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 0.00s
Predicting...


100%|██████████| 24/24 [00:02<00:00, 10.51it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 0.00s
Predicting...
  Predicted in 10.37s
Resampling...
  Resampled in 3.80s
  cropping from (512, 512, 262) to (439, 486, 262)
Resampling...
  Resampled in 5.63s
Predicting...


100%|██████████| 3/3 [00:01<00:00,  2.60it/s]



If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 0.00s
Predicting...
  Predicted in 10.37s
Resampling...
  Resampled in 3.80s
  cropping from (512, 512, 262) to (439, 486, 262)
Resampling...
  Resampled in 5.63s
Predicting...
  Predicted in 13.12s
Resampling...
  Resampled in 1.27s
  cropping from (512, 512, 262) to (260, 244, 210)
Resampling...
  Resampled in 0.40s
Predicting...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 0.00s
Predicting...
  Predicted in 10.37s
Resampling...
  Resampled in 3.80s
  cropping from (51

CompletedProcess(args=['/home/sandy0317/practice_seg/.pixi/envs/default/bin/TotalSegmentator', '--device', 'gpu:4', '-i', '/Public/inference/ToothFairy3/ToothFairy3F_001_0000.nii.gz', '-o', 'teeth', '--task', 'teeth'], returncode=0)

In [ ]:
#查看切割結果
from pathlib import Path
import SimpleITK as sitk
import matplotlib.pyplot as plt

# 以使用者家目錄為基準
base_dir = Path.home()

# 兩張影像的路徑
image_path1 = base_dir / "Public/train/ToothFairy3/imagesTr/ToothFairy3F_001_0000.nii.gz"
image_path2 = Path("teeth/lower_left_second_molar_fdi37.nii.gz")

# 檢查路徑
print("目前工作目錄：", Path.cwd())
print("Image 1：", image_path1, image_path1.exists())
print("Image 2：", image_path2.resolve(), image_path2.exists())

# 先確認檔案存在，再讀取
if not image_path1.exists():
    raise FileNotFoundError(f"找不到第一張影像：{image_path1}")

if not image_path2.exists():
    raise FileNotFoundError(f"找不到第二張影像：{image_path2.resolve()}")

volume1 = sitk.GetArrayFromImage(sitk.ReadImage(str(image_path1)))
volume2 = sitk.GetArrayFromImage(sitk.ReadImage(str(image_path2)))

# 顯示兩張灰階影像
z1 = volume1.shape[0] // 2
z2 = volume2.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(volume1[z1], cmap="gray", origin="lower")
axes[0].set_title(f"ToothFairy3 (Z={z1})")
axes[0].axis("off")

axes[1].imshow(volume2[z2], cmap="gray", origin="lower")
axes[1].set_title(f"FDI 37 (Z={z2})")
axes[1].axis("off")

plt.tight_layout()
plt.show()